# LeetCode 2110 — Number of Smooth Descent Periods of a Stock bằng Python

Link đề: https://leetcode.com/problems/number-of-smooth-descent-periods-of-a-stock/ · Độ khó: **Medium** · Chủ đề: Array, Math, Dynamic Programming

Skeleton LeetCode:

```python
class Solution:
    def getDescentPeriods(self, prices: List[int]) -> int:
        ...
```

Lộ trình:
1. **Brute force**: thử mọi đoạn `[l, r]`, kiểm tra từng bước giảm đúng 1 — `O(n³)`.
2. **Mở rộng từ mỗi `l`** cho tới khi hỏng điều kiện — `O(n²)` tệ nhất.
3. **DP "số đoạn kết thúc tại `i`"**: `run[i] = run[i−1] + 1` nếu `prices[i−1] − prices[i] = 1`, ngược lại `1` — `O(n)`, `O(1)` bộ nhớ (**nộp bài**).
4. **Chia thành các đoạn tối đại** (hai con trỏ): đoạn dài `L` đóng góp `L(L+1)/2`.
5. **Bất biến `prices[i] + i`**: trong một đoạn giảm mượt, `prices[i] + i` không đổi ⇒ `groupby`.

## 1. Mục tiêu học

1. Đọc đúng định nghĩa: mỗi ngày (trừ ngày đầu của đoạn) thấp hơn ngày trước **đúng 1**; đoạn một ngày luôn hợp lệ.
2. Nhận ra tính chất **đóng dưới đoạn con**: mọi đoạn con liên tiếp của một đoạn hợp lệ cũng hợp lệ ⇒ đếm theo **đoạn tối đại**.
3. Viết DP "số đoạn hợp lệ kết thúc tại `i`" — mẫu dùng cho rất nhiều bài đếm mảng con.
4. Dùng công thức `L(L+1)/2` và mẹo bất biến `prices[i] + i`.
5. Phân biệt "giảm đúng 1" với "giảm" hoặc "chênh lệch 1".

## 2. Đề bài & ví dụ

`prices[i]` là giá cổ phiếu ngày `i`. Một **giai đoạn giảm mượt** là một hoặc nhiều ngày liên tiếp sao cho giá mỗi ngày **thấp hơn ngày trước đúng 1** (ngày đầu tiên của giai đoạn được miễn). Đếm số giai đoạn giảm mượt.

**Ví dụ 1:** `[3,2,1,4]` → `7`: `[3]`, `[2]`, `[1]`, `[4]`, `[3,2]`, `[2,1]`, `[3,2,1]`.

**Ví dụ 2:** `[8,6,7,7]` → `4` (`[8,6]` không hợp lệ vì giảm 2).

**Ví dụ 3:** `[1]` → `1`.

**Ràng buộc:** `1 ≤ n ≤ 10⁵`, `1 ≤ prices[i] ≤ 10⁵`.

In [1]:
import random
import time
from itertools import groupby
from typing import List

examples = [([3, 2, 1, 4], 7), ([8, 6, 7, 7], 4), ([1], 1)]
for p, want in examples:
    print(p, "->", want)

[3, 2, 1, 4] -> 7
[8, 6, 7, 7] -> 4
[1] -> 1


## 3. Kiến thức nền

### 3.1 Điều kiện theo từng bước

Đoạn `[l, r]` hợp lệ ⇔ với mọi `j ∈ (l, r]`: `prices[j − 1] − prices[j] = 1`. Gọi bước `j` là **tốt** nếu thỏa. Đoạn hợp lệ ⇔ mọi bước bên trong đều tốt.

In [2]:
p = [5, 4, 3, 5, 4, 2, 1]
good = [p[j - 1] - p[j] == 1 for j in range(1, len(p))]
print("bước j tốt?", list(zip(range(1, len(p)), good)))

bước j tốt? [(1, True), (2, True), (3, False), (4, True), (5, False), (6, True)]


### 3.2 Số đoạn con của một đoạn độ dài `L`

`L` đoạn độ dài 1, `L − 1` đoạn độ dài 2, …, 1 đoạn độ dài `L` ⇒ `L(L+1)/2`.

### 3.3 `itertools.groupby`

Gom các phần tử **liên tiếp** có cùng khóa.

In [3]:
print([(k, len(list(g))) for k, g in groupby("aaabccdd")])

[('a', 3), ('b', 1), ('c', 2), ('d', 2)]


## 4. Ý tưởng nền tảng / trực giác

`[5, 4, 3, 5, 4, 2, 1]`: các bước tốt là `5→4`, `4→3`, `5→4`, `2→1`. Đoạn tối đại (không mở rộng được nữa):

| đoạn tối đại | độ dài `L` | `L(L+1)/2` |
|---|---|---|
| `[5, 4, 3]` | 3 | 6 |
| `[5, 4]` | 2 | 3 |
| `[2, 1]` | 2 | 3 |

Tổng `12`. Cách nhìn DP: `run[i]` = độ dài đoạn hợp lệ dài nhất **kết thúc tại `i`** = số đoạn hợp lệ kết thúc tại `i`:

| `i` | 0 | 1 | 2 | 3 | 4 | 5 | 6 |
|---|---|---|---|---|---|---|---|
| `prices` | 5 | 4 | 3 | 5 | 4 | 2 | 1 |
| `run` | 1 | 2 | 3 | 1 | 2 | 1 | 2 |

`Σ run = 12` ✓.

In [4]:
p = [5, 4, 3, 5, 4, 2, 1]; run = [1] * len(p)
for i in range(1, len(p)):
    if p[i - 1] - p[i] == 1: run[i] = run[i - 1] + 1
print("run =", run, "-> tổng", sum(run))

run = [1, 2, 3, 1, 2, 1, 2] -> tổng 12


## 5. Cách 1 — Brute Force / baseline

In [5]:
def descent_brute(prices):
    n, cnt = len(prices), 0
    for l in range(n):
        for r in range(l, n):
            if all(prices[j - 1] - prices[j] == 1 for j in range(l + 1, r + 1)):
                cnt += 1
    return cnt

for p, want in examples:
    print(descent_brute(p), "kỳ vọng", want)

7 kỳ vọng 7
4 kỳ vọng 4
1 kỳ vọng 1


**Độ phức tạp:** `O(n²)` đoạn × `O(n)` kiểm tra ⇒ `O(n³)`. Space `O(1)`.

**Nút thắt:** nếu `[l, r]` hỏng thì mọi `[l, r']` với `r' > r` cũng hỏng — không cần thử tiếp; và kiểm tra `[l, r]` lặp lại công việc của `[l, r − 1]`.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 (đóng dưới đoạn con):** đoạn hợp lệ ⇒ mọi đoạn con liên tiếp hợp lệ. Ngược lại, đoạn hợp lệ nằm trọn trong **đúng một** đoạn tối đại.

**Insight 2 (đếm theo đầu phải):** số đoạn hợp lệ kết thúc tại `i` bằng `run[i]`:
- *nếu* `prices[i−1] − prices[i] = 1` *thì* `run[i] = run[i−1] + 1` (mọi đoạn kết thúc tại `i − 1` nối thêm `i`, cộng `[i]`);
- *ngược lại* `run[i] = 1` (chỉ `[i]`).

Đáp án `= Σ run[i]`. Chỉ cần giữ `run` của bước trước ⇒ `O(1)` bộ nhớ.

**Insight 3 (bất biến):** trong một đoạn giảm mượt, `prices[i] + i` **không đổi** (giá giảm 1 khi chỉ số tăng 1). Hai ngày kề cùng đoạn ⇔ `prices[i−1] + (i−1) = prices[i] + i`.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Mở rộng từ mỗi `l`

In [6]:
def descent_extend(prices):
    n, cnt = len(prices), 0
    for l in range(n):
        r = l
        while True:
            cnt += 1                                   # [l, r] hợp lệ
            if r + 1 < n and prices[r] - prices[r + 1] == 1:
                r += 1
            else:
                break
    return cnt

for p, want in examples:
    print(descent_extend(p), "kỳ vọng", want)

7 kỳ vọng 7
4 kỳ vọng 4
1 kỳ vọng 1


**Độ phức tạp:** `O(n + đáp án)` — tệ nhất `O(n²)` khi cả mảng là một đoạn giảm mượt (`[10⁵, 10⁵ − 1, …, 1]` ⇒ `5·10⁹`). Space `O(1)`.

### 7.2 Cách 3 — DP số đoạn kết thúc tại `i` (code nộp)

In [7]:
def descent_dp(prices):
    ans = run = 0
    for i, x in enumerate(prices):
        if i > 0 and prices[i - 1] - x == 1:
            run += 1
        else:
            run = 1
        ans += run
    return ans

for p, want in examples:
    print(descent_dp(p), "kỳ vọng", want)

7 kỳ vọng 7
4 kỳ vọng 4
1 kỳ vọng 1


Code nộp LeetCode của Cách 3: xem mục 9.

**Độ phức tạp:** `O(n)` thời gian, `O(1)` bộ nhớ.

### 7.3 Cách 4 — Đoạn tối đại (hai con trỏ) + `L(L+1)/2`

In [8]:
def descent_runs(prices):
    n, i, ans = len(prices), 0, 0
    while i < n:
        j = i
        while j + 1 < n and prices[j] - prices[j + 1] == 1:
            j += 1
        L = j - i + 1
        ans += L * (L + 1) // 2
        i = j + 1
    return ans

for p, want in examples:
    print(descent_runs(p), "kỳ vọng", want)

7 kỳ vọng 7
4 kỳ vọng 4
1 kỳ vọng 1


**Độ phức tạp:** `O(n)` (mỗi chỉ số được `j` đi qua đúng một lần), `O(1)` bộ nhớ.

### 7.4 Cách 5 — `groupby` theo bất biến `prices[i] + i`

In [9]:
def descent_groupby(prices):
    total = 0
    for _, g in groupby(range(len(prices)), key=lambda i: prices[i] + i):
        L = sum(1 for _ in g)
        total += L * (L + 1) // 2
    return total

for p, want in examples:
    print(descent_groupby(p), "kỳ vọng", want, "| khóa:", [x + i for i, x in enumerate(p)])

7 kỳ vọng 7 | khóa: [3, 3, 3, 7]
4 kỳ vọng 4 | khóa: [8, 7, 9, 10]
1 kỳ vọng 1 | khóa: [1]


**Vì sao đúng?** Hai ngày kề `i − 1`, `i` cùng khóa ⇔ `prices[i−1] + i − 1 = prices[i] + i` ⇔ `prices[i−1] − prices[i] = 1` — đúng điều kiện bước tốt. `groupby` gom các ngày **liên tiếp** cùng khóa ⇒ đúng các đoạn tối đại. (Không thể dùng `Counter` theo khóa vì hai đoạn rời nhau có thể trùng khóa — xem Lỗi 4.)

**Độ phức tạp:** `O(n)`.

## 8. TRACE từng bước

Trace Cách 3 trên ví dụ 1 và một mảng dài hơn.

In [10]:
def trace_dp(prices):
    ans = run = 0
    print(f"prices = {prices}")
    print(f"{'i':>2} | giá | giảm đúng 1? | run | ans | các đoạn kết thúc tại i")
    for i, x in enumerate(prices):
        ok = i > 0 and prices[i - 1] - x == 1
        run = run + 1 if ok else 1; ans += run
        print(f"{i:>2} | {x:>3} | {'có' if ok else 'không':>11} | {run:>3} | {ans:>3} | {[prices[i - t + 1:i + 1] for t in range(1, run + 1)]}")
    return ans

trace_dp([3, 2, 1, 4])
trace_dp([5, 4, 3, 5, 4, 2, 1])

prices = [3, 2, 1, 4]
 i | giá | giảm đúng 1? | run | ans | các đoạn kết thúc tại i
 0 |   3 |       không |   1 |   1 | [[3]]
 1 |   2 |          có |   2 |   3 | [[2], [3, 2]]
 2 |   1 |          có |   3 |   6 | [[1], [2, 1], [3, 2, 1]]
 3 |   4 |       không |   1 |   7 | [[4]]
prices = [5, 4, 3, 5, 4, 2, 1]
 i | giá | giảm đúng 1? | run | ans | các đoạn kết thúc tại i
 0 |   5 |       không |   1 |   1 | [[5]]
 1 |   4 |          có |   2 |   3 | [[4], [5, 4]]
 2 |   3 |          có |   3 |   6 | [[3], [4, 3], [5, 4, 3]]
 3 |   5 |       không |   1 |   7 | [[5]]
 4 |   4 |          có |   2 |   9 | [[4], [5, 4]]
 5 |   2 |       không |   1 |  10 | [[2]]
 6 |   1 |          có |   2 |  12 | [[1], [2, 1]]


12

In [11]:
def trace_runs(prices):
    n, i, ans = len(prices), 0, 0
    while i < n:
        j = i
        while j + 1 < n and prices[j] - prices[j + 1] == 1: j += 1
        L = j - i + 1; ans += L * (L + 1) // 2
        print(f"  đoạn tối đại [{i}, {j}] = {prices[i:j + 1]}: L = {L} -> +{L * (L + 1) // 2} (tổng {ans})")
        i = j + 1
    return ans

trace_runs([8, 6, 7, 7]); trace_runs([10, 9, 8, 7, 3, 2, 2, 1])

  đoạn tối đại [0, 0] = [8]: L = 1 -> +1 (tổng 1)
  đoạn tối đại [1, 1] = [6]: L = 1 -> +1 (tổng 2)
  đoạn tối đại [2, 2] = [7]: L = 1 -> +1 (tổng 3)
  đoạn tối đại [3, 3] = [7]: L = 1 -> +1 (tổng 4)
  đoạn tối đại [0, 3] = [10, 9, 8, 7]: L = 4 -> +10 (tổng 10)
  đoạn tối đại [4, 5] = [3, 2]: L = 2 -> +3 (tổng 13)
  đoạn tối đại [6, 7] = [2, 1]: L = 2 -> +3 (tổng 16)


16

In [12]:
def trace_groupby(prices):
    keys = [x + i for i, x in enumerate(prices)]
    print(f"prices = {prices}\nkhóa   = {keys}")
    i = 0
    for k, g in groupby(range(len(prices)), key=lambda t: prices[t] + t):
        idx = list(g); L = len(idx)
        print(f"  nhóm liên tiếp khóa {k}: chỉ số {idx} giá {[prices[t] for t in idx]} -> L = {L}, +{L * (L + 1) // 2}")

trace_groupby([10, 9, 8, 7, 3, 2, 2, 1])

prices = [10, 9, 8, 7, 3, 2, 2, 1]
khóa   = [10, 10, 10, 10, 7, 7, 8, 8]
  nhóm liên tiếp khóa 10: chỉ số [0, 1, 2, 3] giá [10, 9, 8, 7] -> L = 4, +10
  nhóm liên tiếp khóa 7: chỉ số [4, 5] giá [3, 2] -> L = 2, +3
  nhóm liên tiếp khóa 8: chỉ số [6, 7] giá [2, 1] -> L = 2, +3


**Quy về bài chuỗi nhị phân (1513).** Đánh dấu mỗi bước `j` là `1` nếu tốt, `0` nếu xấu, được chuỗi nhị phân độ dài `n − 1`. Một giai đoạn dài `≥ 2` ứng với một **đoạn con toàn `1`** của chuỗi này (các bước bên trong nó). Vậy đáp án `= n + (số đoạn con toàn 1 của chuỗi bước)` — đúng bài 1513 *Number of Substrings With Only 1s*.

In [13]:
def via_binary_steps(prices):
    bits = "".join("1" if prices[j - 1] - prices[j] == 1 else "0" for j in range(1, len(prices)))
    ones = sum(len(b) * (len(b) + 1) // 2 for b in bits.split("0"))      # số đoạn con toàn 1
    return len(prices) + ones, bits

for p, want in examples + [([5, 4, 3, 5, 4, 2, 1], 12)]:
    r, bits = via_binary_steps(p)
    print(f"{p}: chuỗi bước '{bits}' -> {r} (kỳ vọng {want})")
rng = random.Random(9)
for _ in range(500):
    p = [rng.randint(1, 5) for _ in range(rng.randint(1, 15))]
    assert via_binary_steps(p)[0] == descent_dp(p)

[3, 2, 1, 4]: chuỗi bước '110' -> 7 (kỳ vọng 7)
[8, 6, 7, 7]: chuỗi bước '000' -> 4 (kỳ vọng 4)
[1]: chuỗi bước '' -> 1 (kỳ vọng 1)
[5, 4, 3, 5, 4, 2, 1]: chuỗi bước '110101' -> 12 (kỳ vọng 12)


## 9. Code nộp LeetCode

In [14]:
class Solution:
    def getDescentPeriods(self, prices: List[int]) -> int:
        ans = 0
        run = 0
        for i, x in enumerate(prices):
            if i > 0 and prices[i - 1] - x == 1:
                run += 1          # nối các đoạn kết thúc tại i-1 thêm ngày i
            else:
                run = 1           # chỉ đoạn [i]
            ans += run
        return ans

sol = Solution()
for p, want in examples:
    got = sol.getDescentPeriods(p)
    print(got, "OK" if got == want else "SAI")

7 OK
4 OK
1 OK


## 10. Vì sao thuật toán đúng

**Bổ đề (số đoạn kết thúc tại `i`).** Gọi `E(i)` = tập đoạn hợp lệ kết thúc tại `i`. Đoạn `[l, i]` hợp lệ ⇔ `l = i` hoặc (`[l, i−1]` hợp lệ **và** bước `i` tốt). Do đó:
- bước `i` tốt ⇒ `E(i) = {[i]} ∪ {[l, i] : [l, i−1] ∈ E(i−1)}` ⇒ `|E(i)| = |E(i−1)| + 1`;
- bước `i` không tốt ⇒ `E(i) = {[i]}` ⇒ `|E(i)| = 1`.

Biến `run` bằng `|E(i)|` theo quy nạp; mỗi đoạn hợp lệ thuộc đúng một `E(i)` (theo đầu phải) ⇒ `Σ run = ` đáp án. ✓

**Cách 4.** Các đoạn tối đại phân hoạch mảng; một đoạn hợp lệ nằm trong đúng một đoạn tối đại (vì nó không chứa bước xấu nào) và mọi đoạn con của đoạn tối đại đều hợp lệ ⇒ đếm `L(L+1)/2` mỗi đoạn. ✓

**Độ lớn:** tối đa `n(n+1)/2 ≈ 5·10⁹` — vượt `int32` (cần `long` ở C++/Java).

## 11. Phân tích Time / Space

`n ≤ 10⁵`.

| Cách | Time | Space |
|---|---|---|
| 1. Brute force | `O(n³)` | `O(1)` |
| 2. Mở rộng từ `l` | `O(n + ans)` ≤ `O(n²)` | `O(1)` |
| 3. DP `run` | `O(n)` | `O(1)` |
| 4. Đoạn tối đại | `O(n)` | `O(1)` |
| 5. `groupby` | `O(n)` | `O(1)` |

In [15]:
rng = random.Random(1)
n = 10**5
tests = [("ngẫu nhiên", [rng.randint(1, 10**5) for _ in range(n)]),
         ("một đoạn dài", list(range(n, 0, -1))),
         ("bậc thang", [((i // 50) * 100 + 50 - i % 50) for i in range(n)])]
for name, p in tests:
    res = []
    for fname, fn in [("dp", descent_dp), ("runs", descent_runs), ("groupby", descent_groupby), ("submit", Solution().getDescentPeriods)]:
        t0 = time.perf_counter(); r = fn(p); res.append(r)
        print(f"{name:12s} {fname:8s}: {r:>12}  {(time.perf_counter() - t0) * 1000:.1f} ms")
    assert len(set(res)) == 1
t0 = time.perf_counter(); descent_extend(list(range(3000, 0, -1))); print(f"mở rộng từ l, một đoạn dài n = 3000: {time.perf_counter() - t0:.2f}s")

ngẫu nhiên   dp      :       100004  6.4 ms
ngẫu nhiên   runs    :       100004  10.5 ms
ngẫu nhiên   groupby :       100004  22.8 ms
ngẫu nhiên   submit  :       100004  5.7 ms
một đoạn dài dp      :   5000050000  6.8 ms
một đoạn dài runs    :   5000050000  4.8 ms
một đoạn dài groupby :   5000050000  6.0 ms
một đoạn dài submit  :   5000050000  6.7 ms
bậc thang    dp      :      2550000  6.1 ms
bậc thang    runs    :      2550000  5.1 ms
bậc thang    groupby :      2550000  6.4 ms
bậc thang    submit  :      2550000  5.9 ms


mở rộng từ l, một đoạn dài n = 3000: 0.29s


## 12. Bộ test

In [16]:
fns = [descent_extend, descent_dp, descent_runs, descent_groupby, Solution().getDescentPeriods]

for p, want in examples:
    assert descent_brute(p) == want
    for fn in fns:
        assert fn(p) == want

edge = [([5, 4], 3), ([4, 5], 2), ([5, 5], 2), ([5, 3], 2), ([3, 2, 1], 6), ([1, 2, 3], 3),
        ([5, 4, 5, 4], 6), ([2, 1, 2, 1, 0], 9)]
for p, want in edge:
    assert descent_brute(p) == want, p
    for fn in fns:
        assert fn(p) == want, (fn, p)

rng = random.Random(2024)
for _ in range(2000):
    n = rng.randint(1, 25)
    start = rng.randint(1, 30)
    p = [start]
    for _ in range(n - 1):
        p.append(max(1, p[-1] + rng.choice([-1, -1, -1, 0, 1, -2, 3])))   # nhiều bước giảm đúng 1
    want = descent_brute(p)
    for fn in fns:
        assert fn(p) == want, (fn, p)
print("ví dụ + edge + 2000 mảng ngẫu nhiên khớp brute force")

ví dụ + edge + 2000 mảng ngẫu nhiên khớp brute force


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Chấp nhận mọi bước **giảm** (không bắt buộc giảm đúng 1)

In [17]:
def wrong_any_decrease(prices):
    ans = run = 0
    for i, x in enumerate(prices):
        run = run + 1 if i > 0 and prices[i - 1] > x else 1          # SAI: phải giảm ĐÚNG 1
        ans += run
    return ans

for p, want in examples:
    print(f"{p}: sai = {wrong_any_decrease(p)} | đúng = {want}")

[3, 2, 1, 4]: sai = 7 | đúng = 7
[8, 6, 7, 7]: sai = 5 | đúng = 4
[1]: sai = 1 | đúng = 1


Ví dụ 2: `8 → 6` giảm 2 — không hợp lệ. **Sửa:** `prices[i−1] − prices[i] == 1`.

### Lỗi 2 — Dùng `abs(...) == 1` (chấp nhận cả tăng 1)

In [18]:
def wrong_abs(prices):
    ans = run = 0
    for i, x in enumerate(prices):
        run = run + 1 if i > 0 and abs(prices[i - 1] - x) == 1 else 1   # SAI: tăng 1 không phải giảm
        ans += run
    return ans

for p, want in examples + [([1, 2, 3], 3)]:
    print(f"{p}: sai = {wrong_abs(p)} | đúng = {want}")

[3, 2, 1, 4]: sai = 7 | đúng = 7
[8, 6, 7, 7]: sai = 5 | đúng = 4
[1]: sai = 1 | đúng = 1
[1, 2, 3]: sai = 6 | đúng = 3


Ví dụ 2 có `6 → 7` tăng 1 bị chấp nhận. **Sửa:** đúng chiều `prices[i−1] − prices[i] == 1`.

### Lỗi 3 — Không đếm giai đoạn một ngày (reset `run = 0`)

In [19]:
def wrong_reset_zero(prices):
    ans = run = 0
    for i, x in enumerate(prices):
        run = run + 1 if i > 0 and prices[i - 1] - x == 1 else 0       # SAI: [i] cũng là một giai đoạn
        ans += run
    return ans

for p, want in examples:
    print(f"{p}: sai = {wrong_reset_zero(p)} | đúng = {want}")

[3, 2, 1, 4]: sai = 3 | đúng = 7
[8, 6, 7, 7]: sai = 0 | đúng = 4
[1]: sai = 0 | đúng = 1


Đề nói rõ "một ngày cũng là giai đoạn giảm mượt". Reset về `0` đếm thiếu đúng `n`. **Sửa:** reset về `1`.

### Lỗi 4 — Đếm theo khóa `prices[i] + i` bằng `Counter` (gộp các đoạn rời nhau)

In [20]:
from collections import Counter
def wrong_counter_key(prices):
    c = Counter(x + i for i, x in enumerate(prices))                  # SAI: các đoạn khác nhau có thể trùng khóa
    return sum(L * (L + 1) // 2 for L in c.values())

examples_p = [p for p, _ in examples]
for p in examples_p + [[5, 4, 0, 2, 1], [4, 3, 9, 1, 0]]:
    print(f"{p}: khóa {[x + i for i, x in enumerate(p)]} | sai = {wrong_counter_key(p)} | đúng = {descent_brute(p)}")

[3, 2, 1, 4]: khóa [3, 3, 3, 7] | sai = 7 | đúng = 7
[8, 6, 7, 7]: khóa [8, 7, 9, 10] | sai = 4 | đúng = 4
[1]: khóa [1] | sai = 1 | đúng = 1
[5, 4, 0, 2, 1]: khóa [5, 5, 2, 5, 5] | sai = 11 | đúng = 7
[4, 3, 9, 1, 0]: khóa [4, 4, 11, 4, 4] | sai = 11 | đúng = 7


Ba ví dụ của đề không có hai đoạn rời nhau trùng khóa nên lỗi **không lộ**. `[5, 4, 0, 2, 1]` có khóa `5, 5, 2, 5, 5`: hai đoạn `[5, 4]` và `[2, 1]` cùng khóa `5` bị gộp thành một "nhóm" 4 phần tử ⇒ đếm `10 + 1 = 11` thay vì `3 + 1 + 3 = 7`. **Sửa:** chỉ gom các phần tử **liên tiếp** cùng khóa (`groupby`), không gom toàn cục.

### Lỗi 5 — Kiểm tra bước với `prices[i + 1]` và quên phần tử cuối

Viết vòng `for i in range(n − 1)` với điều kiện `prices[i] − prices[i+1] == 1` rồi cộng `run` **trước** khi xét phần tử cuối ⇒ thiếu đóng góp của ngày cuối. Cách an toàn: duyệt `i` từ `0` tới `n − 1`, so với ngày **trước** (như code nộp).

In [21]:
def wrong_forget_last(prices):
    ans, run = 0, 1
    for i in range(len(prices) - 1):
        ans += run                                                     # cộng cho ngày i
        run = run + 1 if prices[i] - prices[i + 1] == 1 else 1
    return ans                                                         # SAI: thiếu ans += run cho ngày cuối

for p, want in examples:
    print(f"{p}: sai = {wrong_forget_last(p)} | đúng = {want}")

[3, 2, 1, 4]: sai = 6 | đúng = 7
[8, 6, 7, 7]: sai = 3 | đúng = 4
[1]: sai = 0 | đúng = 1


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Đếm mảng con liên tiếp thỏa điều kiện **trên từng bước kề**" ⇒ DP `run[i]` (số mảng con hợp lệ kết thúc tại `i`), tổng lại.
- Điều kiện đóng dưới đoạn con ⇒ đếm theo **đoạn tối đại** với `L(L+1)/2`.
- Cấp số cộng với công sai `d` ⇒ bất biến `a[i] − d·i`; ở đây `d = −1` ⇒ `a[i] + i`.
- Họ bài: 413 *Arithmetic Slices*, 1513 *Number of Substrings With Only 1s*, 2348 *Number of Zero-Filled Subarrays*, 1759 *Count Number of Homogenous Substrings*, 795 *Number of Subarrays with Bounded Maximum*.

## 15. Biến thể của bài

1. **Tăng mượt** (tăng đúng 1): đổi dấu.
2. **Giảm đúng `d`**: điều kiện `prices[i−1] − prices[i] = d`, bất biến `prices[i] + d·i`.
3. **413 Arithmetic Slices**: đếm mảng con độ dài `≥ 3` là cấp số cộng (công sai bất kỳ nhưng cố định trong đoạn).
4. **Giai đoạn mượt dài nhất**: `max(run)`.
5. **Cho phép tối đa `c` bước "xấu"** trong giai đoạn: cửa sổ trượt hai con trỏ (Bài 5).

In [22]:
def arithmetic_slices_413(nums):
    """Biến thể 3: số mảng con độ dài >= 3 là cấp số cộng."""
    ans = cur = 0
    for i in range(2, len(nums)):
        cur = cur + 1 if nums[i] - nums[i - 1] == nums[i - 1] - nums[i - 2] else 0
        ans += cur
    return ans

def longest_smooth(prices):
    """Biến thể 4."""
    best = run = 0
    for i, x in enumerate(prices):
        run = run + 1 if i > 0 and prices[i - 1] - x == 1 else 1
        best = max(best, run)
    return best

def descent_step_d(prices, d):
    """Biến thể 2."""
    ans = run = 0
    for i, x in enumerate(prices):
        run = run + 1 if i > 0 and prices[i - 1] - x == d else 1
        ans += run
    return ans

rng = random.Random(15)
for _ in range(300):
    a = [rng.randint(0, 5) for _ in range(rng.randint(1, 12))]
    brute = sum(1 for l in range(len(a)) for r in range(l + 2, len(a))
                if len({a[j] - a[j - 1] for j in range(l + 1, r + 1)}) == 1)
    assert arithmetic_slices_413(a) == brute
    assert descent_step_d(a, 1) == descent_dp(a)
print("413 ví dụ [1,2,3,4] ->", arithmetic_slices_413([1, 2, 3, 4]), "(kỳ vọng 3)",
      "| dài nhất ví dụ 1 ->", longest_smooth([3, 2, 1, 4]), "| giảm đúng 2: [8,6,4,7] ->", descent_step_d([8, 6, 4, 7], 2))

413 ví dụ [1,2,3,4] -> 3 (kỳ vọng 3) | dài nhất ví dụ 1 -> 3 | giảm đúng 2: [8,6,4,7] -> 7


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute force | định nghĩa | `O(n³)` | `O(1)` | kiểm chứng |
| 2. Mở rộng từ `l` | dừng khi hỏng | `O(n²)` tệ nhất | `O(1)` | minh họa |
| 3. DP `run` | đếm theo đầu phải | `O(n)` | `O(1)` | **nộp bài** |
| 4. Đoạn tối đại | `L(L+1)/2` | `O(n)` | `O(1)` | cần thông tin đoạn |
| 5. `groupby` khóa | bất biến | `O(n)` | `O(1)` | ngắn gọn |

## 17. Bài tập (từ dễ đến khó)

**Bài 1 (đoán output):** `[4, 3, 2, 1]`? `[1, 1, 1]`? `[10, 9, 9, 8, 7]`?

**Bài 2 (tính tay):** `[6, 5, 4, 6, 5, 4, 3]`: viết bảng `run` và tổng.

**Bài 3 (điền TODO):** hoàn thiện `maximal_runs(prices)` trả về danh sách `(l, r)` của các đoạn giảm mượt tối đại.

**Bài 4 (tự viết lại):** viết `count_length_at_least(prices, m)` = số giai đoạn giảm mượt có **độ dài ≥ m**.

**Bài 5 (mini challenge):** "giai đoạn gần mượt": cho phép **tối đa `c` bước** không giảm đúng 1 bên trong giai đoạn. Đếm số giai đoạn trong `O(n)` bằng hai con trỏ. Kiểm chứng bằng brute force. (Biến thể tự đặt — đối chiếu lại đề gốc nếu dùng làm đề.)

In [23]:
def maximal_runs(prices):
    # TODO: hai con trỏ i, j như Cách 4, thu (i, j)
    pass

## 18. Gợi ý 3 mức

**Bài 1:** (1) Một đoạn tối đại dài 4 ⇒ `10`. (2) Mỗi ngày một đoạn ⇒ `3`. (3) Đoạn `[10, 9]` và `[9, 8, 7]` ⇒ `3 + 6 = 9`.

**Bài 2:** (1) `run = [1, 2, 3, 1, 2, 3, 4]`. (2) Tổng `16`. (3) Kiểm tra bằng `L(L+1)/2`: `6 + 10 = 16`.

**Bài 3:** (1) `i = 0`. (2) Mở rộng `j` khi bước tốt. (3) Thu `(i, j)`, đặt `i = j + 1`.

**Bài 4:** (1) Đoạn tối đại dài `L` có `max(0, L − m + 1)` đoạn con độ dài đúng `m`… nhưng cần độ dài `≥ m`: `Σ_{t=m}^{L} (L − t + 1)`. (2) Hoặc DP: cộng `max(0, run − m + 1)` tại mỗi `i`. (3) `m = 1` cho đáp án gốc.

**Bài 5:** (1) Đầu trái `l`, đếm `bad` = số bước xấu trong `(l, r]`. (2) Khi `bad > c`, tăng `l` (nếu bước `l + 1` xấu thì `bad −= 1`). (3) Cộng `r − l + 1` mỗi `r`.

## 19. Đáp án tham khảo

In [24]:
# Bài 1
assert [Solution().getDescentPeriods(p) for p in ([4, 3, 2, 1], [1, 1, 1], [10, 9, 9, 8, 7])] == [10, 3, 9]
# Bài 2
assert trace_dp([6, 5, 4, 6, 5, 4, 3]) == 16

# Bài 3
def maximal_runs_ans(prices):
    n, i, out = len(prices), 0, []
    while i < n:
        j = i
        while j + 1 < n and prices[j] - prices[j + 1] == 1: j += 1
        out.append((i, j)); i = j + 1
    return out

# Bài 4
def count_length_at_least_ans(prices, m):
    ans = run = 0
    for i, x in enumerate(prices):
        run = run + 1 if i > 0 and prices[i - 1] - x == 1 else 1
        ans += max(0, run - m + 1)
    return ans

rng = random.Random(41)
for _ in range(500):
    p = [rng.randint(1, 6) for _ in range(rng.randint(1, 15))]; m = rng.randint(1, 4)
    runs = maximal_runs_ans(p)
    assert sum((r - l + 1) * (r - l + 2) // 2 for l, r in runs) == descent_brute(p)
    brute = sum(1 for l in range(len(p)) for r in range(l + m - 1, len(p))
                if all(p[j - 1] - p[j] == 1 for j in range(l + 1, r + 1)))
    assert count_length_at_least_ans(p, m) == brute
print("Bài 3 ví dụ:", maximal_runs_ans([5, 4, 3, 5, 4, 2, 1]), "| Bài 4 (m = 2):", count_length_at_least_ans([5, 4, 3, 5, 4, 2, 1], 2))
print("Bài 1-4: OK")

prices = [6, 5, 4, 6, 5, 4, 3]
 i | giá | giảm đúng 1? | run | ans | các đoạn kết thúc tại i
 0 |   6 |       không |   1 |   1 | [[6]]
 1 |   5 |          có |   2 |   3 | [[5], [6, 5]]
 2 |   4 |          có |   3 |   6 | [[4], [5, 4], [6, 5, 4]]
 3 |   6 |       không |   1 |   7 | [[6]]
 4 |   5 |          có |   2 |   9 | [[5], [6, 5]]
 5 |   4 |          có |   3 |  12 | [[4], [5, 4], [6, 5, 4]]
 6 |   3 |          có |   4 |  16 | [[3], [4, 3], [5, 4, 3], [6, 5, 4, 3]]
Bài 3 ví dụ: [(0, 2), (3, 4), (5, 6)] | Bài 4 (m = 2): 5
Bài 1-4: OK


In [25]:
# Bài 5 — cho phép tối đa c bước xấu
def near_smooth(prices, c):
    n = len(prices); l = 0; bad = 0; ans = 0
    for r in range(n):
        if r > 0 and prices[r - 1] - prices[r] != 1:
            bad += 1                                    # bước r xấu
        while bad > c:
            if prices[l] - prices[l + 1] != 1:          # bước l+1 rời khỏi cửa sổ
                bad -= 1
            l += 1
        ans += r - l + 1
    return ans

def near_smooth_brute(prices, c):
    n = len(prices)
    return sum(1 for l in range(n) for r in range(l, n)
               if sum(prices[j - 1] - prices[j] != 1 for j in range(l + 1, r + 1)) <= c)

rng = random.Random(55)
for _ in range(1000):
    p = [rng.randint(1, 6) for _ in range(rng.randint(1, 15))]; c = rng.randint(0, 3)
    assert near_smooth(p, c) == near_smooth_brute(p, c)
    if c == 0: assert near_smooth(p, 0) == descent_dp(p)
print("Bài 5 ví dụ 2 với c = 0, 1, 2:", [near_smooth([8, 6, 7, 7], c) for c in range(3)])

Bài 5 ví dụ 2 với c = 0, 1, 2: [4, 7, 9]


## 20. Template nhớ nhanh + Checklist

```python
# Đếm mảng con mà mọi bước kề thỏa điều kiện P(a[i-1], a[i])
ans = run = 0
for i, x in enumerate(a):
    run = run + 1 if i > 0 and P(a[i - 1], x) else 1   # số mảng con hợp lệ kết thúc tại i
    ans += run
# Ở bài này: P(u, v) = (u - v == 1)
```

**Checklist trước khi submit:**
- [ ] Giảm **đúng 1** (`prices[i−1] − prices[i] == 1`), không phải "giảm" hay `abs`.
- [ ] Reset `run = 1` (giai đoạn một ngày hợp lệ).
- [ ] Không bỏ sót ngày cuối.
- [ ] Nếu dùng khóa `prices[i] + i`: chỉ gom phần tử **liên tiếp**.
- [ ] Kết quả có thể tới `≈ 5·10⁹`.